In [28]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

# Data Preparation

In [29]:
sc = StandardScaler()
ml_df = pd.DataFrame(columns=["game_state", "next_move", "winner"])


df = pd.read_csv("data.csv")
df["moves"] = df["moves"].apply(lambda x: list(map(int, x.split(","))))    

rows = []
weights = []

for _, d in df.iterrows():        
    A = np.zeros((6,7))         
    turn = d["played_first"]
    winner = d["winner"]
    moves = d["moves"]
    n = len(moves)

    for i, col in enumerate(moves):
        weight = 1
        winner_view = 1

        if turn == winner:
            winner_view = -1
            if i == n-1:
                weight = 5
            elif i == n-2:
                weight = 3
            elif i == n-3:
                weight = 2

        game_state = A.copy()

        for row in range(5, -1, -1):
            if A[row][col] == 0:
                A[row][col] = turn
                break

        game_state = (game_state * -turn).flatten().tolist()

        rows.append({
            "id": id,
            "game_state": game_state,
            "next_move": col,
            "winner": winner_view
        })

        weights.append(weight)
        turn = -turn
        

ml_df = pd.DataFrame(rows)



# Model Training

In [30]:
sc = StandardScaler()
model = keras.Sequential([
    layers.Input(shape=(42,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax")
])

X = np.array(ml_df["game_state"].tolist())    
X = sc.fit_transform(X)
y = np.array(ml_df["next_move"])    
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.fit(X,y,sample_weight=np.array(weights), epochs=100, batch_size=512, verbose=1)
print(len(X))


Epoch 1/100
14/14 [==============================] - 0s 796us/step - loss: 2.5032 - accuracy: 0.1918
Epoch 2/100
14/14 [==============================] - 0s 707us/step - loss: 2.3490 - accuracy: 0.2897
Epoch 3/100
14/14 [==============================] - 0s 659us/step - loss: 2.2728 - accuracy: 0.3164
Epoch 4/100
14/14 [==============================] - 0s 711us/step - loss: 2.2158 - accuracy: 0.3292
Epoch 5/100
14/14 [==============================] - 0s 677us/step - loss: 2.1647 - accuracy: 0.3427
Epoch 6/100
14/14 [==============================] - 0s 664us/step - loss: 2.1171 - accuracy: 0.3572
Epoch 7/100
14/14 [==============================] - 0s 669us/step - loss: 2.0706 - accuracy: 0.3599
Epoch 8/100
14/14 [==============================] - 0s 586us/step - loss: 2.0278 - accuracy: 0.3736
Epoch 9/100
14/14 [==============================] - 0s 652us/step - loss: 1.9862 - accuracy: 0.3792
Epoch 10/100
14/14 [==============================] - 0s 642us/step - loss: 1.9425 - accura

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

14/14 [==============================] - 0s 660us/step - loss: 1.9041 - accuracy: 0.3995
Epoch 12/100
14/14 [==============================] - 0s 619us/step - loss: 1.8635 - accuracy: 0.4077
Epoch 13/100
14/14 [==============================] - 0s 625us/step - loss: 1.8267 - accuracy: 0.4197
Epoch 14/100
14/14 [==============================] - 0s 700us/step - loss: 1.7902 - accuracy: 0.4286
Epoch 15/100
14/14 [==============================] - 0s 710us/step - loss: 1.7567 - accuracy: 0.4448
Epoch 16/100
14/14 [==============================] - 0s 657us/step - loss: 1.7268 - accuracy: 0.4483
Epoch 17/100
14/14 [==============================] - 0s 631us/step - loss: 1.6979 - accuracy: 0.4575
Epoch 18/100
14/14 [==============================] - 0s 606us/step - loss: 1.6658 - accuracy: 0.4671
Epoch 19/100
14/14 [==============================] - 0s 652us/step - loss: 1.6416 - accuracy: 0.4727
Epoch 20/100
14/14 [==============================] - 0s 728us/step - loss: 1.6154 - accuracy: 

# Model Prediction

In [34]:
def check_winner(A, c, t):
    r = -1
    for row in range(5,-1,-1):
        if A[row][c] == 0:
            A[row][c] = -1
            r = row
            break
    dirs = [
        (1, 0),
        (0, 1),
        (1, 1),
        (1, -1),
    ]

    for dr, dc in dirs:
        count = 1        
        rr, cc = r + dr, c + dc
        while 0 <= rr < 6 and 0 <= cc < 7 and A[rr][cc] == t:
            count += 1
            rr += dr
            cc += dc

        rr, cc = r - dr, c - dc
        while 0 <= rr < 6 and 0 <= cc < 7 and A[rr][cc] == t:
            count += 1
            rr -= dr
            cc -= dc

        if count >= 4:
            return True

    return False

def predict(X):
    X = sc.transform([X])    
    return model.predict(X, verbose=0)[0]

def getNextMove(game_state):
    probs = predict(game_state)
    A = np.array(game_state).reshape(6,7)
    valid = [c for c in range(7) if A[0][c] == 0]    
    for col in valid:
        B = A.copy()        
        if check_winner(B,col,-1):
            return int(col)
    for col in valid:
        B = A.copy()        
        if check_winner(B,col,1):
            return int(col)
    valid_probs = [(c, probs[c]) for c in valid]
    print(valid_probs)
    valid_probs.sort(key=lambda x: x[1], reverse=True)
    next_move = valid_probs[np.random.randint(0,min(1,len(valid_probs)))][0]
    return int(next_move)

# Examples

In [44]:
# Game 1
A = [[0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0]]
A = np.array(A)
next_move = getNextMove(A.flatten())
print("Next Move:", next_move)
print("Winner:", check_winner(A, next_move, -1))
print()

# Game 2
A = [[0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, -1, 1, 0, 0, 0],
     [0, 0, -1, 1, 0, 0, 0]]
A = np.array(A)
next_move = getNextMove(A.flatten())
print("Next Move:", next_move)
print("Winner:", check_winner(A, next_move, -1))
print()

# Game 3
A = [[0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 0],
     [0, 0, -1, 0, 0, 0, 0],
     [0, 0, -1, 1, 0, 0, 0],
     [0, 0, -1, 1, 1, 0, 0]]
A = np.array(A)
next_move = getNextMove(A.flatten())
print("Next Move:", next_move)
print("Winner:", check_winner(A, next_move, -1))
print()


[(0, 0.08693506), (1, 0.042015154), (2, 0.09384416), (3, 0.6195005), (4, 0.05411176), (5, 0.05665114), (6, 0.04694226)]
Next Move: 3
Winner: False

[(0, 0.0026122942), (1, 0.01226484), (2, 0.9008789), (3, 0.051957507), (4, 0.006075785), (5, 0.006317754), (6, 0.019892847)]
Next Move: 2
Winner: False

Next Move: 2
Winner: True

